# DeFi-LSA D2 Real Data Collection on Kaggle

This notebook clones a GitHub repository, loads credentials from Kaggle Secrets, installs the package, writes runtime-only source configuration, runs tests, performs connectivity checks, and launches D2 collection.

## Required Kaggle Secrets
- `GITHUB_REPO_URL`: HTTPS URL of the public repository, or a URL that does not embed credentials.
- `GITHUB_TOKEN`: optional, only for a private repository.
- `ETH_RPC_URL`: Ethereum mainnet JSON-RPC endpoint.
- `ETHERSCAN_API_KEY`: Etherscan API key.
- `COINGECKO_API_KEY`: optional CoinGecko Demo key. Keyless mode may be used at lower rate limits.

Enable Internet in the Kaggle notebook settings before running.


In [ ]:
from pathlib import Path
import os, json, subprocess, sys
from kaggle_secrets import UserSecretsClient

secrets = UserSecretsClient()
def get_secret(name, required=False):
    try:
        value = secrets.get_secret(name)
    except Exception:
        value = None
    if required and not value:
        raise RuntimeError(f"Missing required Kaggle Secret: {name}")
    return value

repo_url = get_secret("GITHUB_REPO_URL", required=True)
github_token = get_secret("GITHUB_TOKEN")
os.environ["ETH_RPC_URL"] = get_secret("ETH_RPC_URL", required=True)
os.environ["ETHERSCAN_API_KEY"] = get_secret("ETHERSCAN_API_KEY", required=True)
cg_key = get_secret("COINGECKO_API_KEY")
if cg_key:
    os.environ["COINGECKO_API_KEY"] = cg_key

WORK = Path("/kaggle/working")
REPO = WORK / "defi-lsa"
OUTPUT = WORK / "defi-lsa-output"
OUTPUT.mkdir(parents=True, exist_ok=True)
print("Secrets loaded; values are not displayed.")


In [ ]:
import urllib.request
urllib.request.urlopen("https://github.com", timeout=15).read(64)
print("Internet connectivity: OK")


In [ ]:
import subprocess, os
if REPO.exists():
    subprocess.run(["git", "-C", str(REPO), "pull", "--ff-only"], check=True)
else:
    clone_url = repo_url
    env = os.environ.copy()
    if github_token and repo_url.startswith("https://github.com/"):
        clone_url = repo_url.replace("https://", f"https://x-access-token:{github_token}@", 1)
    subprocess.run(["git", "clone", "--depth", "1", clone_url, str(REPO)], check=True, env=env)
print(subprocess.check_output(["git", "-C", str(REPO), "rev-parse", "HEAD"], text=True).strip())


In [ ]:
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-U", "pip"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", f"{REPO}[dev,parquet,aws]"], check=True)
print("Package installation: OK")


In [ ]:
runtime_config = {
  "ethereum_rpc_url": os.environ["ETH_RPC_URL"],
  "etherscan_api_url": "https://api.etherscan.io/v2/api",
  "coingecko_api_url": "https://api.coingecko.com/api/v3",
  "gdelt_enabled": True,
  "request_timeout_seconds": 30,
  "maximum_attempts": 5,
  "requests_per_second": 2.0,
  "initial_log_chunk_size": 2000,
  "minimum_log_chunk_size": 10,
  "maximum_log_chunk_size": 20000
}
config_path = OUTPUT / "runtime_sources.json"
config_path.write_text(json.dumps(runtime_config, indent=2), encoding="utf-8")
print(f"Runtime configuration written to {config_path}; do not publish this file because it contains the RPC URL.")


In [ ]:
subprocess.run([sys.executable, "-m", "pytest", "-q", str(REPO / "tests")], cwd=REPO, check=True)


In [ ]:
# This command must perform real probes after Epic E2 is implemented.
subprocess.run([
    sys.executable, "-m", "defi_lsa", "connectivity",
    "--event", "euler",
    "--config", str(config_path),
    "--source", "rpc"
], cwd=REPO, check=True)


## Smoke collection
Run a small real-data collection before a full event window. The CLI must be wired by Epic E3. If the current repository only writes a summary, stop and implement E3 rather than interpreting the command as a successful collection.


In [ ]:
subprocess.run([
    sys.executable, "-m", "defi_lsa", "collect",
    "--event", "euler",
    "--stage", "logs",
    "--source", "rpc",
    "--config", str(config_path),
    "--output", str(OUTPUT),
    "--resume"
], cwd=REPO, check=True)


In [ ]:
subprocess.run([
    sys.executable, "-m", "defi_lsa", "audit",
    "--event", "euler",
    "--stage", "d2",
    "--output", str(OUTPUT)
], cwd=REPO, check=True)


## Full event runs
Uncomment only after the smoke audit reports `GO` or an approved `CONDITIONAL_GO`, and after Euler/Curve/Kyber registries contain complete windows, incident legs, token mappings, and public-source queries.


In [ ]:
EVENTS = ["euler", "curve", "kyber"]
RUN_FULL = False
if RUN_FULL:
    for event in EVENTS:
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "collect",
            "--event", event,
            "--stage", "all",
            "--source", "auto",
            "--config", str(config_path),
            "--output", str(OUTPUT),
            "--resume"
        ], cwd=REPO, check=True)
        subprocess.run([
            sys.executable, "-m", "defi_lsa", "audit",
            "--event", event,
            "--stage", "d2",
            "--output", str(OUTPUT)
        ], cwd=REPO, check=True)


In [ ]:
# Sanity-check generated artifacts without exposing secrets.
for path in sorted(OUTPUT.rglob("*")):
    if path.is_file() and path != config_path:
        print(path.relative_to(OUTPUT), path.stat().st_size)
